# **TAS (Tele Assistance System) Stochastic Modelling**
NOTE: _[DASA CASE STUDY 1]_

## **Summary**

This notebook is focused on three main objectives:
1. summarizing the key aspects of the Tele Assistance System (TAS) architecture and its adaptive capabilities in the context of telehealth services for chronic patients.
2. Modelling the TAS architecture using appropriate design notations and tools to visualize its components and interactions.
3. Stochasticly Simulate the TAS behavior under different scenarios to evaluate its performance and adaptability with Queue Network (QN) models.

The results will be used to evaluate the Dimensional Analysis for Software Architecture (DASA) methodology, its software tool (PyDASA) and its effectiveness in modelling and Quality Scenarios (QS) trade-off in self-adaptive-systems (SAS).

---

## **Software Architecture**
- TAS (Tele Assistance System) operates in a dynamic environment where service quality, availability, and user needs frequently change.
- The TAS is further subdivided into Controller and Target System subsystem components.
- The Controller is responsible for managing the overall system behavior, while the Target System focuses on executing specific tasks related to patient care.
- The TAS target systems follows a Service-oriented architecture (SOA) pattern.
- The TAS Controller follows a MAPE-K (Monitor-Analyze-Plan-Execute-Knowledge) feedback loop for self-adaptation.
- Adaptations focus on maintaining **reliability**, **performance**, and **compliance** with patient care standards (5 specific scenarios).
- ActivFORMS provides the runtime framework for model-based adaptation using runtime models, simulations, and verified decision-making.

---

_**NOTE: MORE DETAILS ON THE ARCHITECTURE IN THE ANALYTICAL MODELLING NOTEBOOK!.**_

---

## **Queue Network Model**

<svg viewBox="0 0 4650 2000" width="1400" height="650">
    <!-- SVG content -->
    <image href="assets/cs1/img/04A - Queue Network.svg" alt="queue-net-diagram" />
    <div align="center"><em>Image 1. TAS Queue Network Diagram.</em></div>
</svg>

## **Code**

_**SUMMARY:**_

This code is for the stochastic simulation of the Case Study (TAS) Queue Network Model and is structured as follows:
1. Analytical Queue Network (QN) model
2. Importing necessary libraries and modules.
3. Loading QN default configuration.
4. Simulating the QN analytically (Stochastic Process).
5. Plotting the QN with the obtained metrics.
6. Loading QN 'optimal' configuration.
7. Simulating the QN optimally (Stochastic Process).
8. Plotting the optimal QN with the obtained metrics.
9. Saving the results.
10. Comparing the simulation results (Default Vs. Optimal)
11. Visualizing the results.
12. Generating a summary report.

### **Necessary Imports**

In [ ]:
# -*- coding: utf-8 -*-
# Native imports
import os
import re
import sys
import time
from typing import Union

# Third-party imports
import numpy as np
import pandas as pd

# import queue stochastic network + models packages
from src.model.analytical import calculate_net_metrics
# from src.simulation.network import QueueNode
# from src.simulation.network import job_generator, job
from src.simulation.network import simulate_network

# import plot functions + grahics
from src.view.plots import plot_queue_network
from src.view.plots import plot_net_comparison
from src.view.plots import plot_net_difference
from src.view.plots import plot_nodes_heatmap
from src.view.plots import plot_nodes_diffmap

### **Function Definitions**

In [ ]:
# Simple formatter for console output

def fmt(val: Union[int, float, np.number]) -> Union[str, np.ndarray]:
    """Format a number to 4 decimal places for console output.

    Args:
        val (Union[int, float, np.number, np.ndarray]): The value to format.

    Returns:
        Union[str, np.ndarray]: The formatted value as a string or an array of strings.
    """
    if isinstance(val, (int, float, np.number)):
        if np.isnan(val) or np.isinf(val):
            return str(val)
        return f"{val:.4f}"
    elif isinstance(val, np.ndarray):
        return np.array([fmt(x) for x in val])
    return val

In [ ]:
# Load configuration from a CSV file
def load(path: str, fname: str) -> pd.DataFrame:
    """Load configuration from a CSV file.

    Args:
        path (str): The directory path where the CSV file is located.
        fname (str): The name of the CSV file to load.

    Returns:
        pd.DataFrame: A DataFrame containing the configuration data.
            CSV format:
                - node: <node_id>
                - miu: <mean_service_time>
                - c: <service_channels>
                - K: <buffer_capacity | max_queue_length>
                - lambda0: <initial_arrival_rate>
                - L0: <initial_queue_length>
                - pm: <matrix_routing_probabilities>
    """
    # path = os.path.dirname(__file__)
    _file_path = os.path.join(path, fname)
    print(f"Loading configuration from: {_file_path}")
    df = pd.read_csv(_file_path)
    return df

In [ ]:
# save dataframes in CSV files
def save(path: str, fname: str, data: pd.DataFrame) -> None:
    """Save a DataFrame to a CSV file.

    Args:
        path (str): The directory path where the CSV file will be saved.
        fname (str): The name of the CSV file to save.
        data (pd.DataFrame): The DataFrame containing the data to save.
    """
    # path = os.path.dirname(__file__)
    _file_path = os.path.join(path, fname)
    print(f"Saving data to: {_file_path}")
    data.to_csv(_file_path, index=False)

In [ ]:
# path = os.path.dirname(__file__)\
PATH = os.getcwd()
print(f"Notebook path: {PATH}")

In [ ]:
# Folder names
asset_folder = "assets"
docs_folder = "docs"
img_folder = "img"
data_folder = "data"
report_folder = "reports"
results_folder = "results"
cs_folder = "cs1"

In [ ]:
# setting case study data folder
file_path = os.path.join(PATH, data_folder, cs_folder)
print(f"Data path: {file_path}")

### **Queue Model**

#### **Stochastic Simulation**

##### **Base Configuration**

In [ ]:
# Load configuration with mixed queue models
dflt_cs_cfg = load(file_path, "default_qn_model.csv")
print("Queue Network Configuration:")
dflt_cs_cfg.head()

In [ ]:
# extract parameters from the configuration DataFrame
# and casting them to proper types
nodes = list(dflt_cs_cfg["node"].values.astype(int))
names = list(dflt_cs_cfg["name"].values)
types = list(dflt_cs_cfg["type"].values)
mius = list(dflt_cs_cfg["miu"].values)
lambda_zs = list(dflt_cs_cfg["lambda0"].values)
n_servers = list(dflt_cs_cfg["s"].values.astype(int))
kaps = list(dflt_cs_cfg["K"].values.astype(float))

# Convert K=0=nan to understandable infinite capacity -> None
for i in range(len(kaps)):
    if np.isnan(kaps[i]):
        kaps[i] = None
    else:
        kaps[i] = float(kaps[i])

# Convert string representations of arrays to actual numpy arrays
# and create routing matrix P
prob = []
for pm_str in dflt_cs_cfg["pm"].values:
    pm_values = pm_str.strip("[]").split(",")
    pm_values = [float(val) for val in pm_values]
    prob.append(pm_values)
P = np.array((prob))

In [ ]:
# simulate the network
dflt_simul_nd_metrics = simulate_network(mius,
                                         lambda_zs,
                                         P,
                                         n_servers,
                                         kaps,
                                         sim_time=10000,)
                                        #  warm_up=1000,
                                        #  n_replications=10)

In [ ]:
# then network metrics
dflt_simul_net_metrics = calculate_net_metrics(dflt_simul_nd_metrics)
dflt_simul_net_metrics["nodes"] = len(list(dflt_simul_nd_metrics["node"]))

In [ ]:
print("\n--- Stochastic Network Simulation (Node Metrics) ---")
# print(dflt_simul_nd_metrics)

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "dflt_stochastic_node_metrics.csv", dflt_simul_nd_metrics)
dflt_simul_nd_metrics.head()

In [ ]:
print("\n--- Stochastic Network Simulation (Network-wide Metrics) ---")
# print(dflt_simul_net_metrics)

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "dflt_stochastic_net_metrics.csv", dflt_simul_net_metrics)
dflt_simul_net_metrics.head()

In [ ]:
# plotting the queue network with metrics on each node
# data table column names
col_names =[
    "Component",
    "$\mathbf{\\lambda}$ [req/s]",
    "$\mathbf{\\mu}$ [req/s]",
    "$\mathbf{\\rho}$",
    "$\mathbf{L}$ [req]",
    "$\mathbf{L_q}$ [req]",
    "$\mathbf{W}$ [s/req]",
    "$\mathbf{W_q}$ [s/req]"
]

node_names = dflt_cs_cfg["name"].values.tolist()
print(f"Datatable column names: {col_names}")
print(f"Node names: {node_names}")  

# selecting images folder
file_path = os.path.join(PATH, results_folder, cs_folder, img_folder)
print(f"Data path: {file_path}")

# Plot the queue network
plot_queue_network(P,
                   dflt_simul_net_metrics,
                   dflt_simul_nd_metrics,
                   node_names,
                   col_names,
                   file_path,
                   "dflt_stochastic_qn_diagram.png")

##### **Optimized Configuration**

In [ ]:
# Load configuration with mixed queue models
file_path = os.path.join(PATH, data_folder, cs_folder)
opti_cs_cfg = load(file_path, "optimal_qn_model.csv")
print("Queue Network Configuration:")
# print(opti_cs_cfg)
opti_cs_cfg.head()

In [ ]:
# extract parameters from the configuration DataFrame
# and casting them to proper types
nodes = list(opti_cs_cfg["node"].values.astype(int))
names = list(opti_cs_cfg["name"].values)
types = list(opti_cs_cfg["type"].values)
mius = list(opti_cs_cfg["miu"].values)
lambda_zs = list(opti_cs_cfg["lambda0"].values)
n_servers = list(opti_cs_cfg["s"].values.astype(int))
kaps = list(opti_cs_cfg["K"].values.astype(float))

# Convert K=0=nan to understandable infinite capacity -> None
for i in range(len(kaps)):
    if np.isnan(kaps[i]):
        kaps[i] = None
    else:
        kaps[i] = float(kaps[i])

# Convert string representations of arrays to actual numpy arrays
# and create routing matrix P
prob = []
for pm_str in opti_cs_cfg["pm"].values:
    pm_values = pm_str.strip("[]").split(",")
    pm_values = [float(val) for val in pm_values]
    prob.append(pm_values)
P = np.array((prob))

In [ ]:
# Simulate the network stochastically
# first node metrics
opti_simul_nd_metrics = simulate_network(mius,
                                         lambda_zs,
                                         P,
                                         n_servers,
                                         kaps,
                                         sim_time=1000,)
#  warm_up=1000,
#  n_replications=10)

In [ ]:
# then network metrics
opti_simul_net_metrics = calculate_net_metrics(opti_simul_nd_metrics)
opti_simul_net_metrics["nodes"] = len(list(opti_simul_nd_metrics["node"]))

In [ ]:
print("\n--- Stochastic Network Simulation (Node Metrics) ---")
# print(opti_simul_nd_metrics)

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "opti_stochastic_node_metrics.csv", opti_simul_nd_metrics)
opti_simul_nd_metrics.head()

In [ ]:
print("\n--- Stochastic Network Simulation (Network-wide Metrics) ---")
# print(opti_simul_net_metrics)
opti_simul_net_metrics.head()

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "opti_stochastic_net_metrics.csv", opti_simul_net_metrics)
opti_simul_net_metrics.head()

In [ ]:
# plotting the queue network with metrics on each node
node_names = opti_cs_cfg["name"].values.tolist()
print(f"Node names: {node_names}")  

# selecting images folder
file_path = os.path.join(PATH, results_folder, cs_folder, img_folder)
print(f"Data path: {file_path}")

# Plot the queue network
plot_queue_network(P,
                   opti_simul_net_metrics,
                   opti_simul_nd_metrics,
                   node_names,
                   col_names,
                   file_path,
                   "opti_stochastic_qn_diagram.png")

## **Results**

### **Compare Results**

In [ ]:
opti_simul_net_metrics.columns

In [ ]:
# prep comparison
dsnm = dflt_simul_net_metrics
osnm = opti_simul_net_metrics

In [ ]:
print("--- Comparing Stochastic Network Metrics ---")
# comparing network metrics
# diff_simul_net_metrics = opti_simul_net_metrics - dflt_simul_net_metrics
# delta_simul_net_metrics = diff_simul_net_metrics / dflt_simul_net_metrics
delta_simul_net_metrics = (osnm - dsnm) / dsnm.abs()

src_col_names = delta_simul_net_metrics.columns.tolist()

tgt_col_names = [
    "delta_avg_miu",
    "delta_avg_rho",
    "delta_L_net",
    "delta_Lq_net",
    "delta_W_net",
    "delta_Wq_net",
    "delta_throughput",
    "delta_nodes",
]

rename_map = dict(zip(src_col_names, tgt_col_names))
# print(rename_map)

# rename comparison columns
delta_simul_net_metrics.rename(columns=rename_map,
                               inplace=True)
delta_simul_net_metrics.head()

In [ ]:
# preparing data comparison
important_cols = [
    "node",
    "lambda",
    "miu",
    "rho",
    "L",
    "Lq",
    "W",
    "Wq"
]

dsnm = dflt_simul_nd_metrics[important_cols]
osnm = opti_simul_nd_metrics[important_cols]

In [ ]:
# comparing node network metrics
print("--- Comparing Stochastic Node/Component Metrics ---")
# extra data columns
extra_cols = [
    "node",
    "name",
    "type",
]

delta_simul_nd_metrics = (osnm - dsnm) / dsnm.abs()

src_col_names = delta_simul_nd_metrics.columns.tolist()

tgt_col_names = [
    "node",
    "delta_lambda",
    "delta_miu",
    "delta_rho",
    "delta_L",
    "delta_Lq",
    "delta_W",
    "delta_Wq",
]

rename_map = dict(zip(src_col_names, tgt_col_names))

# rename comparison columns
delta_simul_nd_metrics.rename(columns=rename_map,
                              inplace=True)

# adding node ID data
for col in extra_cols:
    if col in opti_cs_cfg.columns:
        delta_simul_nd_metrics[col] = opti_cs_cfg[col].values

delta_simul_nd_metrics.head()

### **Saving Results**

In [ ]:
# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path,
     "delta_stochastic_node_metrics.csv",
     delta_simul_nd_metrics)

In [ ]:
# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path,
     "delta_stochastic_net_metrics.csv",
     delta_simul_net_metrics)

## **Analysis**

### **Graph Analysis**

In [ ]:
# selecting images folder
print("--- Configuring folder path for plot parameters ---")
file_path = os.path.join(PATH, results_folder, cs_folder, img_folder)
print(f"Data path: {file_path}")

In [ ]:
print("--- Charting Overall Configuration Comparisons ---")
metrics = dflt_simul_net_metrics.columns.tolist()
labels = [
    "$\\mathbf{\\mu}$ [req/s]",
    "$\\mathbf{\\rho}$ [%]",
    "$\\mathbf{L_{net}}$ [req]",
    "$\\mathbf{L_{q_{net}}}$ [req]",
    "$\\mathbf{W_{net}}$ [s/req]",
    "$\\mathbf{W_{q_{net}}}$ [s/req]",
    "$\\mathbf{Th_{net}}$ [req]",
    "$\\mathbf{n}$ [comp]",
]

for m, l in zip(metrics, labels):
    print(f"{m:18} : {l}")

In [ ]:
# Plot the metric comparison
plot_net_comparison([dflt_simul_net_metrics, opti_simul_net_metrics],
                    ["Default Configuration", "Adaptation Configuration"],
                    metrics,
                    labels,
                    "Metric Comparison: Before and after Adaptation",
                    file_path,
                    "net_stochastic_metric_comparison.png")

In [ ]:
print("--- Charting Overall Configuration differences ---")
metrics = delta_simul_net_metrics.columns.tolist()
labels = [
    "$\\mathbf{\\overline{\\Delta \\mu}}$ [req/s]",
    "$\\mathbf{\\overline{\\Delta \\rho}}$ [n.a.]",
    "$\\mathbf{\\overline{\\Delta L}_{net}}$ [req]",
    "$\\mathbf{\\overline{\\Delta L}_{q_{net}}}$ [req]",
    "$\\mathbf{\\overline{\\Delta W}_{net}}$ [s/req]",
    "$\\mathbf{\\overline{\\Delta W}_{q_{net}}}$ [s/req]",
    "$\\mathbf{\\overline{\\Delta Th}_{net}}$ [req]",
    "$\\mathbf{\\overline{\\Delta n}}$ [comp]",
]

for m, l in zip(metrics, labels):
    print(f"{m:18} : {l}")

In [ ]:
# Plot the metric differences
plot_net_difference(delta_simul_net_metrics,
                    metrics,
                    labels,
                    "Change between configurations after adaptation.",
                    file_path,
                    "net_stochastic_metric_differences.png")

In [ ]:
print("--- Charting Component Queue-Network Comparative Heatmap ---")
# Define metrics for the heatmap X-axis
metrics = delta_simul_nd_metrics.select_dtypes(include="number")
metrics = metrics.columns.tolist()
if "node" in metrics:
    metrics.remove("node")

# define the labels for the heatmap X-axis alias
labels = [
    "$\\mathbf{\\Delta\\lambda}$ [req/s]",
    "$\\mathbf{\\Delta \\mu}$ [req/s]",
    "$\\mathbf{\\Delta \\rho}$ [n.a.]",
    "$\\mathbf{\\Delta L_{net}}$ [req]",
    "$\\mathbf{\\Delta L_{q_{net}}}$ [req]",
    "$\\mathbf{\\Delta W_{net}}$ [s/req]",
    "$\\mathbf{\\Delta W_{q_{net}}}$ [s/req]",
]

# define the node names for the heatmap Y-axis
node_names = delta_simul_nd_metrics["name"].values.tolist()
print(f"Node names: {node_names}")

for m, l in zip(metrics, labels):
    print(f"{m:18} : {l}")

In [ ]:
print("--- Preparing data for heatmaps ---")
dflt_simul_nd_metrics["name"] = node_names
opti_simul_nd_metrics["name"] = node_names
print(dflt_simul_nd_metrics.columns.tolist())
print(opti_simul_nd_metrics.columns.tolist())

In [ ]:
print("--- Charting Component Queue-Network Configuration Heatmap ---")
metrics = dflt_simul_nd_metrics.select_dtypes(include="number")
metrics = metrics.columns.tolist()
if "node" in metrics:
    metrics.remove("node")

labels = [
    # "$\\mathbf{n}$ [comp]",
    "$\\mathbf{\\lambda}$ [req/s]",
    "$\\mathbf{\\mu}$ [req/s]",
    "$\\mathbf{\\rho}$ [%]",
    "$\\mathbf{L}$ [req]",
    "$\\mathbf{L_{q}}$ [req]",
    "$\\mathbf{W}$ [s/req]",
    "$\\mathbf{W_{q}}$ [s/req]",
]

# define the node names for the heatmap Y-axis
node_names = delta_simul_nd_metrics["name"].values.tolist()
print(f"Node names: {node_names}")

for m, l in zip(metrics, labels):
    print(f"{m:18} : {l}")

In [ ]:
# removing numeric metrics that Im not interested in
not_interesting = [
    "node",
    "type",
    "L_littles",
    "Lq_littles",
    "Jobs_Served",
    "Jobs_Blocked",
    "Blocking_Prob"
]
print(metrics)
metrics = [m for m in metrics if m not in not_interesting]
print(metrics)

In [ ]:
if "name" not in metrics:
    metrics.append("name")
print(metrics)
print(labels)
print(len(metrics), len(labels))

dsnm = dflt_simul_nd_metrics[metrics]
osnm = opti_simul_nd_metrics[metrics]
# quitar las columnas 'L_littles', 'Lq_littles', 'Jobs_Served', 'Jobs_Blocked', 'Blocking_Prob'

In [ ]:
if "name" in metrics:
    metrics.remove("name")
print(metrics)
print(labels)
print(len(metrics), len(labels))

In [ ]:
dsnm.head()

In [ ]:
opti_simul_nd_metrics.columns

In [ ]:
plot_nodes_heatmap([dsnm, osnm],
                   ["Default Configuration", "Adaptation Configuration"],
                   node_names,
                   metrics,
                   labels,
                   "Component Performance Comparison Between Configurations",
                   "name",
                   file_path,
                   "nodes_stochastic_metric_heatmap.png")

In [ ]:
print("--- Charting Component Queue-Network Differential Heatmap ---")
# Define metrics for the heatmap X-axis
metrics = delta_simul_nd_metrics.select_dtypes(include="number")
metrics = metrics.columns.tolist()
if "node" in metrics:
    metrics.remove("node")

# define the labels for the heatmap X-axis alias
labels = [
    "$\\mathbf{\\Delta\\lambda}$ [%]",
    "$\\mathbf{\\Delta \\mu}$ [%]",
    "$\\mathbf{\\Delta \\rho}$ [%]",
    "$\\mathbf{\\Delta L_{net}}$ [%]",
    "$\\mathbf{\\Delta L_{q_{net}}}$ [%]",
    "$\\mathbf{\\Delta W_{net}}$ [%]",
    "$\\mathbf{\\Delta W_{q_{net}}}$ [%]",
]

# define the node names for the heatmap Y-axis
node_names = delta_simul_nd_metrics["name"].values.tolist()
print(f"Node names: {node_names}")

for m, l in zip(metrics, labels):
    print(f"{m:18} : {l}")

In [ ]:
plot_nodes_diffmap(delta_simul_nd_metrics,
                   node_names,
                   metrics,
                   labels,
                   "Component Performance Change: Before and after Adaptation",
                   "name",
                   file_path,
                   "nodes_stochastic_metric_diffmap.png")

: 

## **Conclusion**

## **Future Work**

## **References & Sources**
<!-- TODO fix the references, links and details -->
1. [Queueing Theory](https://en.wikipedia.org/wiki/Queueing_theory)
2. [Dimensional Analysis](https://en.wikipedia.org/wiki/Dimensional_analysis)
3. [Simulation in Healthcare](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6466220/)

---

# **HASTA AKI!!!**